In [ ]:
# Import the necessary libraries
import pandas as pd
from openai import OpenAI

In [ ]:
# Load the data
df = pd.read_csv("data/transcriptions.csv")
df.head()

In [ ]:
# Initialize the OpenAI client
client = OpenAI()

## Start coding here, use as many cells as you need

messages_system = {
    "role": "system",
    "content": (
        "You have been provided with an anonymized dataset of medical transcriptions organized by specialty. "
        "You get it in function extract_patient_data. Make a table containing age, medical specialty, treatment and  International Classification of Diseases (ICD) code."
        "Respond ONLY in valid JSON format as required by the response_format parameter. "
        "if only treatment is matching the ICD-10 Code"
    )
}

functions_tools = [
    {
        "type": "function",
        "function": {
            "name": "extract_patient_data",
            "description": "Get the Patient Data from the body of the input text",
            "parameters": {
                "type": "object",
                "properties": {
                    "age": {
                        "type": "string",
                        "description": "get the age or year old in just the number{transcription}"
                    },
                    "medical_specialty": {
                        "type": "string",
                        "description": "medical specialty in {medical_specialty}"
                    },
                    "treatment": {
                        "type": "string",
                        "description": "get the treatment in {transcription}"
                    },
                    "icd_code": {
                        "type": "string",
                        "description": "get International Classification of Diseases (ICD) code"
                    }
                }
            }
        }
    }
]


df_structured = pd.DataFrame()
for i in range(len(df)):
    messages = [
        messages_system,
        {
            "role": "user",
            "content": (
                "Extract the patient's age, medical specialty, and treatment from the following transcription. "
                "Return your answer in JSON format.\n\n"     
                f"Medical Specialty: {medical_specialty.iloc[i]}\n"
                f"Transcription: {transcription.iloc[i]}"
            )
        }
    ]
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=functions_tools,
        response_format={"type": "json_object"}
    )
    

# The arguments are a JSON string, so we need to parse them first
    arguments = response.choices[0].message.tool_calls[0].function.arguments
    data = json.loads(arguments)
    
# Wrap the dictionary in a list to create a DataFrame with one row
    df_structured = df_structured.append([data])
df_structured.head()